In [10]:
import torch
import random
import numpy as np
from collections import deque
import time

In [11]:
# state will be the 1d board with 0s for empty, 1s for Xs and 2s for Os

In [12]:
# TO DO

# ADD SYMMETRY CODE - data augment??
# layer norm for bigger boards

In [13]:
class Board:
    def __init__(self, board_size=(3,3), n=3):

        self.rows, self.cols = board_size
        self.board_size = self.rows * self.cols # 1D board size

        self.n = n  # n in a row

        self.board = np.zeros((self.rows*self.cols)) # 1 for Os, 2 for Xs

        self.win_lines = self.calc_win_lines()

    
    def reset(self):
        self.board = np.zeros((self.board_size))
        return self.board

    
    def step(self, action, player): 
        
        self.board[action] = player     # update board with new move
        done, winner = self.is_terminal(self.board, player)   # check if game ended

        # after a step the current player can only have won or drew (never lose obv)
        if done and winner==player:
            return self.board, 1, done  # current player won
        else:
            return self.board, 0, done  # game hasnt ended or draw
        
    def is_terminal(self, board, last_player):  # 1 or 2

        # bool arrays (1 or 0)
        player_board = (board == last_player) 

        # defaults if not done
        done = False
        winner = 0
        
        for line in self.win_lines:
            if player_board[line].all():
                winner = last_player
                done = True
                break
        
        # draw check must be afterwards (if no 3s found AND full board) -> could win on final move
        if not done and sum(board==0) == 0:   
            winner = 0
            done = True

        return done, winner
        
    def calc_win_lines(self):
        win_lines = []

        # win rows
        for i in range(self.rows):  # down the rows
            for j in range(self.n-1, self.cols):  #left to right
                idx = i*self.cols+j
                win_lines.append(np.arange(idx-self.n+1, idx+1).tolist())
    
        # win cols
        for j in range(self.cols):  # across the cols
            for i in range(self.n-1, self.rows):  # up to down
                idx = i*self.cols+j
                win_lines.append(np.arange(idx-self.cols*(self.n-1),idx+1, self.cols).tolist())   # start, stop, step

        # win main diags
        for i in range(self.rows-self.n+1):
            for j in range(self.cols-self.n+1):
                start = i * self.cols + j
                win_lines.append([start + k * (self.cols+1) for k in range(self.n)])

        # win other diags
        for i in range(self.rows-self.n+1):
            for j in range(self.n-1, self.cols):
                start = i * self.cols + j
                win_lines.append([start + k * (self.cols-1) for k in range(self.n)])

        return win_lines
    
    

In [14]:
# TO DO
class DQNAgent:
    def __init__(self, board_size_2D, board_size, DDQN=True, model_type='MLP', gamma=0.99, alpha=1e-4, epsilon=1.0, epsilon_decay=0.9995, min_epsilon=0.0, batch_size=64):

        self.rows, self.cols = board_size_2D
        self.board_size = board_size   # 1d
        self.DDQN = DDQN    # use DDQN or just DQN
        self.model_type = model_type    # use CNN or MLP
        self.gamma = gamma
        self.alpha = alpha
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon
        self.batch_size = batch_size
        
        # init models
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.init_model()
        self.target_network = self.init_model()
        self.target_network.load_state_dict(self.model.state_dict()) # must match main network initially
        self.target_network.eval()

        # Sample pool
        self.datas = deque(maxlen=40000) # 10k


    def init_model(self):

        # output_size: 1D board_size (q_val for every pos)

        # MLP input_size is (batch, c*board_size)    2 channel (1 board per player)
        if self.model_type == "MLP":
            model = torch.nn.Sequential(   
                torch.nn.Linear(2*self.board_size, 256),
                torch.nn.ReLU(),
                torch.nn.Linear(256, 256),
                torch.nn.ReLU(),
                torch.nn.Linear(256, self.board_size),
            )
        
        # CNN input_size is (batch, c, rows, cols)    2 channel (1 board per player)
        elif self.model_type == "CNN": 
            model = torch.nn.Sequential(   
                torch.nn.Conv2d(2, 8, kernel_size=(3,3), padding=1),  # padding 1 with kernel 3x3 preserves board size
                torch.nn.ReLU(),
                torch.nn.Conv2d(8, 64, kernel_size=(3,3), padding=1),
                torch.nn.ReLU(),
                torch.nn.Flatten(),
                # out = (in + 2*padding - kernel_size) // stride + 1
                torch.nn.Linear(64 * self.rows * self.cols, self.board_size),
            )

        return model.to(self.device) ##!

    
    # convert board state to model input state
    def get_model_input(self,state):

        # SPLIT into 2 PATHS
        # 1 - tensors come from get_value/get_target - already on device
        # 2 - np arrays come from get_action

        # TORCH PATH
        if isinstance(state, torch.Tensor):
            board1 = (state==1).float() # needed since bool arrays cant pass through model
            board2 = (state==2).float()

            if self.model_type == "MLP":
                x = torch.cat([board1, board2], dim=1)
            
            elif self.model_type == "CNN":
                board1 = board1.view(-1, self.rows, self.cols)  # no copy view instead of reshape
                board2 = board2.view(-1, self.rows, self.cols)
                x = torch.stack([board1, board2], dim=1) # (b,c,rows,cols)
            
            return x    # already on device


        # NUMPY PATH - from get_action
        # state arrives as np - either (board_size,) from env or (batch,board_size) from get_sample()
        if state.ndim == 1:
            state = state[np.newaxis, :]    # add batch dim if needed
            
        board1 = (state==1)
        board2 = (state==2)

        if self.model_type == "MLP":
            x = np.concatenate([board1,board2], axis=1) # (batch, 2*board_size)

        elif self.model_type == "CNN":
            board1 = board1.reshape(-1, self.rows, self.cols) # (b, rows, cols)
            board2 = board2.reshape(-1, self.rows, self.cols)
            x = np.stack([board1, board2], axis=1)     # (b, c, rows, cols)
        
        return torch.FloatTensor(x).to(self.device) ##!
    

    def get_action(self, state, greedy=False):  # use greedy=True for eval

        # state from env - always np

        # Epsilon Greedy
        valid_actions = (state == 0)    # bool mask

        # Exploration - return random action
        if not greedy and np.random.uniform(0,1) < self.epsilon:
            action = np.random.choice(np.flatnonzero(valid_actions))    # indices where valid_actions!=0 (board==0) -> rand idx
            return action
        
        # Exploitation
        with torch.no_grad(): #why?
            x = self.get_model_input(state)
            q_vals = self.model(x).squeeze(0).cpu().numpy()   # remove batch dim, convert to np
            q_vals[~valid_actions] = -np.inf    # invalid masked (0s in valid)
            best_action = np.argmax(q_vals)
            return best_action

    
    # Get a batch of samples
    def get_sample(self):
    
        samples = random.sample(self.datas, self.batch_size)

        # transposes a list of tuples into separate tuples using C?
        states, actions, rewards, next_states, dones = zip(*samples)

        # Convert to numpy then tensor (much faster than looping)
        states = torch.FloatTensor(np.array(states)).to(self.device)
        actions = torch.LongTensor(np.array(actions)).to(self.device)
        rewards = np.array(rewards, dtype=np.float32)                   # stays on cpu since doesnt go through model
        next_states = torch.FloatTensor(np.array(next_states)).to(self.device)
        dones = np.array(dones, dtype=bool)                             # stays on cpu since doesnt go through model

        return states, actions, rewards, next_states, dones # (batch, ...)


    def store(self, state, action, reward, next_state, done):
        self.datas.append((state.copy(), action, reward, next_state.copy(), done))


    def get_value(self, states, actions): # actions already valid checked
    
        # states and actions both tensors from get_sample

        # states already torch but get model input expects np
        # round trip cost negligible for small boards - would need to amend get_model_input to also accept tensor

        x = self.get_model_input(states)
        q_vals = self.model(x)   
        value = q_vals[range(len(actions)), actions]   # keep actions as torch

        # returns q-values for all actions
        return value    # return is torch for loss_fn


    def get_target(self, rewards, next_states, dones):

        # rewards, next_states, dones from get_sample()

        valid_actions = (next_states==0)

        with torch.no_grad():
            x = self.get_model_input(next_states)

            # Double DQN
            if self.DDQN:
                q_vals = self.model(x).masked_fill(~valid_actions, float('-inf'))
                next_actions = q_vals.argmax(dim=1) # (batch,) of max indices 

                # target network evaluates main network's actions
                target_vals = self.target_network(x).masked_fill(~valid_actions, float('-inf'))
                target = target_vals[range(len(next_actions)), next_actions]  # ()

            # standard DQN
            else:
                target_vals = self.target_network(x).masked_fill(~valid_actions, float('-inf'))
                target = target_vals.max(dim=1).values  # (batch,)  max returns (value, idx)?? [0]?


        rewards = torch.FloatTensor(rewards).to(self.device)
        dones = torch.BoolTensor(dones).to(self.device)

        # zero out future return for terminal states
        target[dones] = 0   
        target = target * self.gamma + rewards

        return target   # return tensor for loss_fn to work


    def update_target_network(self):
        # copy main network params to target network
        self.target_network.load_state_dict(self.model.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)
        


In [15]:
# RULES OF THUMB

# turnover rate = steps_per_epoch / maxlen
# aim for 2% ish

# times_each_experience_trained = (updates_per_epoch * batch_size) / maxlen
# aim for 2-4 times per experience

# update target network every 1-5k gradient steps

In [16]:
# per Epoch:

# 20 episodes * 10ish moves per game = 200 steps
# maxlen of experience buffer: 10,000 -> 2%

# 100 training steps * batch size (64) = 6400
# 6400 / maxlen (10k) = 0.64 times per experience

# target network update: 
# gradient updates per epoch = 100
# 100 * 50 (updates every n epochs) = 5000

In [17]:
def flip_board(state, player):
    if player == 1:
        return state.copy() # why copy needed?
    
    flipped = state.copy()
    flipped[state==1] = 2
    flipped[state==2] = 1
    return flipped

In [18]:
# MAIN

env = Board(board_size=(9,9), n=5)          # 1e-3?
agent = DQNAgent((env.rows,env.cols), env.board_size, DDQN=True, alpha=1e-4, epsilon_decay=0.9996, model_type='CNN') # pass 1d board size

epochs = 6001
min_buffer = 10000 #30000

optimizer = torch.optim.Adam(agent.model.parameters(), lr=agent.alpha)
loss_fn = torch.nn.MSELoss()
agent.model.train()

GAMES_PER_EPOCH = 10 #20

print(f"training on: {agent.device}")

for e in range(epochs):

    # decouple training from experience collecting

    steps = 0   # track game steps

    # play some full games to collect experiences
    for _ in range(GAMES_PER_EPOCH):         # 20 games - each around 10 steps
        state = env.reset()
        done = False
        player = 1

        prev_state = {1: None, 2: None}
        prev_action = {1: None, 2: None}

        while not done:
            flipped_state = flip_board(state, player)   # flip board perspective if player==2 (just for agent)
            action = agent.get_action(flipped_state)
            
            # store NON TERMINAL transition
            if prev_state[player] is not None:
                agent.store(prev_state[player], prev_action[player], 0, flipped_state, False) # SARS + done
            
            # save state & action for next time it's player's turn
            prev_state[player] = flipped_state
            prev_action[player] = action

            # step in env
            next_state, reward, done = env.step(action, player=player)
            steps += 1

            # store TERMINAL transition
            if done:
                # transition for player who just moved - (they won or drew)
                flipped_next_player = flip_board(next_state, player) # new added - needed? - doesnt make actual diff since zeroed out
                agent.store(flipped_state, action, reward, flipped_next_player, True)

                other_player = 2 if player==1 else 1
                flipped_next_other = flip_board(next_state, other_player) # new added - needed?
                agent.store(prev_state[other_player], prev_action[other_player], -reward, flipped_next_other, True) # why also next state?
                break

            state = next_state
            player = 2 if player==1 else 1

        
    # do multiple gradient updates from buffer
    if len(agent.datas) >= min_buffer:
        #print("Min buffer reached...")
        for _ in range(100): #200
            # Sample a batch of data
            s, a, r, ns, d = agent.get_sample()

            # Compute the value and target for the batch of samples
            value = agent.get_value(s, a)
            target = agent.get_target(r, ns, d)

            # Update parameters
            loss = loss_fn(value, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # update target
    if e % 10 == 0:#30
        agent.update_target_network()

    if e % 500 == 0:
        print(f"Epoch: {e}   Eps:{agent.epsilon}    Avg Steps: {steps/GAMES_PER_EPOCH}")
        #if e > 1000:
        torch.save(agent.model.state_dict(), "DQN_ninarow1.pth")

    # reduce exploration every epoch
    agent.decay_epsilon()



training on: cuda
Epoch: 0   Eps:1.0    Avg Steps: 52.8
Epoch: 500   Eps:0.8186979957674545    Avg Steps: 60.3
Epoch: 1000   Eps:0.6702664082736469    Avg Steps: 57.4
Epoch: 1500   Eps:0.5487457650838841    Avg Steps: 58.8
Epoch: 2000   Eps:0.4492570580600538    Avg Steps: 61.6
Epoch: 2500   Eps:0.3678058530181487    Avg Steps: 58.7
Epoch: 3000   Eps:0.30112191469749694    Avg Steps: 63.3
Epoch: 3500   Eps:0.24652790804449892    Avg Steps: 44.7
Epoch: 4000   Eps:0.20183190421677408    Avg Steps: 37.2
Epoch: 4500   Eps:0.16523937546420142    Avg Steps: 49.3
Epoch: 5000   Eps:0.13528114551440729    Avg Steps: 57.0
Epoch: 5500   Eps:0.11075440269777033    Avg Steps: 57.5
Epoch: 6000   Eps:0.09067440751108606    Avg Steps: 29.0


In [19]:
# helper

def print_board(board, board_size):

    BOARD = [" ", "X", "O"]
    rows, cols = board_size

    #for i in range(rows):
        #print(i, end=" ")
    #print()

    for i in range(rows):
        for j in range(cols):
            print(f"|{BOARD[board[i,j]]}", end="")
        print("|")
        print("-" * (2*cols+1))
        
    print()

In [20]:
# eval

env = Board(board_size=(9,9), n=5)
agent = DQNAgent((env.rows,env.cols), env.board_size, DDQN=True, model_type='CNN')    # DQN or DDQN

# load trained model...
agent.model.load_state_dict(torch.load("DQN_ninarow1.pth"))
agent.model.eval()

board = np.zeros(env.board.size, dtype=np.int8)
rows, cols = env.rows, env.cols

print_board(board.reshape(rows,cols), board_size=(rows,cols))

while True:
    # player1 - you
    valid = False
    
    while not valid:
        row = -1
        col = -1
        while row<0 or row>=rows:
            row = int(input("enter row: "))
        while col<0 or col>=cols:
            col = int(input("enter col: "))

        flat_idx = row * cols + col
        if board[flat_idx] == 0:
            board[flat_idx] = 1
            valid = True
        else:
            print("cant place piece here - position already occupied!")

    print_board(board.reshape(rows,cols), board_size=(rows,cols))
    
    done, winner = env.is_terminal(board, last_player=1)
    if done and winner!=0:
        print(f"player {winner} won")
        break
    elif done and winner==0:
        print("draw")
        break

    # player2
    flipped_state = flip_board(board, player=2)  # Add this line
    action = agent.get_action(flipped_state, greedy=True)  # take best action
    board[action] = 2

    print_board(board.reshape(env.rows,env.cols), board_size=(env.rows,env.cols))

    done, winner = env.is_terminal(board,last_player=2)
    if done and winner!=0:
        print(f"player {winner} won")
        break
    elif done and winner==0:
        print("draw")
        break

    

| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------

| | | | | | | | | |
-------------------
| |X| | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------

| | | | | | | | | |
-------------------
| |X|O| | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
-------------------
| | | | | | | | | |
------------------

ValueError: invalid literal for int() with base 10: ''

In [ ]:
state_key = env.get_state_key(np.array([1,0,0,1,1,0,2,0,2]))
print(agent.q_table[state_key],"\n")


[ 0.         -0.20539    -0.284851    0.          0.          0.171
  0.          0.99484622  0.        ] 

